# PDCP · N09 — LiteJudge: does decoupled (PDSG) training remove cue shortcuts? (H7, contribution C4)
**Notebook 9 of 12.** Kaggle: **GPU T4 x2** (or P100 — detected automatically) · Internet **On** · **Save & Run All (Commit)**. Expected ≈ 3–4 h on T4 x2 (≈ 5–6 h on P100). Resumable.

**Inputs:** `kbs-n07-output` (LiteJudge drawings + images) · `kbs-n07-v2-output` (N07b: `lj_pairs_full.parquet` with the balanced flag `in_bal`, new images) · `kbs-n3-output` (VLM evaluation pairs: main, E1b Mooney, NU/ELD/XR counterfactuals) · `kbs-n6-part2` (MEL counterfactuals + null edits).

**Model.** Siamese judge: a shared image backbone gives each drawing a scalar stress score s(x); for a pair, u = s(B) − s(A) (u > 0 → A has lower stress). u is antisymmetric, so the judge has no position bias.
Ordinal head (CORAL; classes B-lower < same < A-lower; learned symmetric margin τ) + auxiliary regression of the 5 layout metrics. Indifference zone |Δstress| < 0.02 = "same". Random dihedral augmentation (all metrics are invariant to it), dihedral test-time averaging. No model selection: the last epoch is evaluated.

**Main factor:** share of PDSG (decoupled, balanced) pairs in each training epoch, p ∈ {0, 10, 30, 50, 100 %}, fixed epoch size, ConvNeXt-T, 3 seeds.
**Ablations:** backbone (DINOv2-S, ResNet-18), frozen DINOv2 (linear probe of features), indifference zone (0.01 / 0.05), no auxiliary heads, input resolution 320.

**Evaluation sets:** LiteJudge test graphs (natural pairs, balanced PDSG pairs, extra PDSG pairs incl. Δstress 0.15) and the **exact VLM evaluation pairs** (N03 main + E1b, N03/N06 counterfactuals incl. null edits) re-rendered at the training resolution.
**Shortcut measures:** standardized cue weights (regression of u on Δstress and Δcues), cue share = Σ|β_cue| / (Σ|β_cue| + |β_stress|), lure gap at Δstress = 0, congruent − conflict accuracy gap, counterfactual flip rates vs the null-edit floor.

**Outputs** (`/kaggle/working/n09_output/`): `runs/*.json|log` (curves), `preds/*.parquet`, `ckpt/` (2 models), `eval_pairs_meta.parquet`, `lj_runs.csv`, `lj_mix_summary.csv`, `lj_betas.csv`, `lj_cf_effects.csv`, `n09_report.json`, figures `fig_lj_*.png`, `lj_train.py`, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
N07_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n07-output"
N07B_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n07-v2-output"
N03_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output"
N06_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n6-part2"
RESUME_ROOT = None                  # path of a previous N09 output (saved as dataset) to skip finished runs
OUTPUT_DIR = "/kaggle/working/n09_output"
CACHE = "/tmp/lj_cache"             # image cache (not saved)
IMG = 224                           # input resolution (images rendered at 384 and resized)
EPOCHS = 12
EPOCH_PAIRS = 6000                  # pairs per epoch (fixed, whatever the PDSG share)
BATCH = 64                          # pairs per step (= 128 images)
LR_HEAD = 1e-3                      # backbone lr: ConvNeXt 1e-4, DINOv2 3e-5, ResNet-18 3e-4
IZ = 0.02                           # indifference zone |Δstress| < IZ -> "same"
LAM_AUX = 0.1                       # weight of auxiliary metric heads
TTA = 8                             # dihedral test-time averaging (1..8)
MIXES = [0.0, 0.1, 0.3, 0.5, 1.0]   # PDSG share of training pairs
ABL_MIXES = [0.0, 0.3]              # mixes used for the ablations
SEEDS = [0, 1, 2]
IZ_ABL = [0.01, 0.05]
PRETRAINED = True
SESSION_HOURS = 11.3
MIN_LEFT_MIN = 25                   # do not start a new run with less time left than this
N_CPU = 4
FAST = False                        # True = tiny smoke test

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib
T0 = time.time()
for d in ["runs", "preds", "ckpt"]: os.makedirs(f"{OUTPUT_DIR}/{d}", exist_ok=True)
os.makedirs(CACHE, exist_ok=True)
for p in ["timm", "joblib", "pyarrow"]:
    try: importlib.import_module(p)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=True)
import numpy as np, pandas as pd, torch, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed
N_GPU = torch.cuda.device_count()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout if N_GPU else "no GPU")
assert N_GPU >= 1 or FAST, "Set Accelerator = GPU T4 x2 (or P100)"
def find(root, name):
    f = sorted(glob.glob(f"{root}/**/{name}", recursive=True)); assert f, f"{name} not found under {root}"; return os.path.dirname(f[0])
N07, N07B, N03, N06 = find(N07_ROOT, "lj_drawings.parquet"), find(N07B_ROOT, "lj_pairs_full.parquet"), find(N03_ROOT, "pairs.parquet"), find(N06_ROOT, "mel_cf_pairs.parquet")
print("N07:", N07, "\nN07b:", N07B, "\nN03:", N03, "\nN06:", N06)
shutil.copy(f"{N07}/pdcp_utils.py", f"{OUTPUT_DIR}/pdcp_utils.py"); sys.path.insert(0, OUTPUT_DIR); import pdcp_utils as U; print("utils", U.__version__)
D7 = pd.read_parquet(f"{N07}/lj_drawings.parquet").assign(root=N07)
D7b = pd.read_parquet(f"{N07B}/lj_drawings_new.parquet").assign(root=N07B)
LJP = pd.read_parquet(f"{N07B}/lj_pairs_full.parquet")
PR = pd.read_parquet(f"{N03}/pairs.parquet"); DR = pd.read_parquet(f"{N03}/drawings.parquet")
MCP = pd.read_parquet(f"{N06}/mel_cf_pairs.parquet"); MCD = pd.read_parquet(f"{N06}/mel_cf_drawings.parquet")
PR = PR[PR.set.isin(["main", "e1b_mooney", "counterfactual"])]
if FAST:
    LJP = LJP[LJP.graph.isin([0, 18, 21])]
    PR = pd.concat([PR[PR.set == s].head(60) for s in ["main", "e1b_mooney"]] + [PR[(PR.set == "counterfactual") & PR.base_pair.isin(PR[PR.set == "main"].head(60).pair_id)]])
    MCP = MCP[MCP.base_pair.isin(PR.pair_id)]
print(len(LJP), "LiteJudge pairs |", PR.set.value_counts().to_dict(), "| N06 counterfactual pairs", len(MCP))

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

N07: /kaggle/input/datasets/resadussalehinrafsan/kbs-n07-output/n07_output 
N07b: /kaggle/input/datasets/resadussalehinrafsan/kbs-n07-v2-output/n07b_output 
N03: /kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output/n03_output 
N06: /kaggle/input/datasets/resadussalehinrafsan/kbs-n6-part2/n06_output
utils 0.1.3
17152 LiteJudge pairs | {'counterfactual': 2839, 'main': 1926, 'e1b_mooney': 289} | N06 counterfactual pairs 1546


## 1. Image registry and cache (N07/N07b PNGs; N03/N06 drawings re-rendered at 384 px with the training renderer)

In [3]:
M5 = ["ksm", "nu", "eld", "xr", "mel"]
need7 = set(LJP.drawing_A) | set(LJP.drawing_B)
R7 = pd.concat([D7, D7b])
R7 = R7[R7.drawing_id.isin(need7)].assign(src="png", path=lambda d: d.root + "/" + d.png)
need3 = set(PR.drawing_A) | set(PR.drawing_B) | set(MCP.drawing_B)
R3 = DR[DR.drawing_id.isin(need3)].assign(src="render", mel=np.nan)
R6 = MCD[MCD.drawing_id.isin(set(MCP.drawing_A))].assign(src="render")
REG = pd.concat([R7, R3, R6], ignore_index=True).drop_duplicates("drawing_id").reset_index(drop=True)
assert set(REG.drawing_id) >= need7 | need3 | set(MCP.drawing_A), "missing drawings"
SIZES = sorted({IMG, 320})
def load_one(rec):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, pdcp_utils as U
    from PIL import Image
    mel = rec["mel"]
    if rec["src"] == "render":
        gc = U.GraphCache(U.make_graph(rec["family"], int(rec["n"]), int(rec["graph_seed"])))
        X = np.array(rec["coords"]).reshape(-1, 2)
        im = U.render_png(gc, X, None, res=384)
        if not np.isfinite(mel): mel = U.mean_edge_length(X, gc.E)
    else:
        im = Image.open(rec["path"])
    im = im.convert("L")
    return [np.asarray(im.resize((s, s), Image.LANCZOS), dtype=np.uint8) for s in SIZES], float(mel)
cols = ["drawing_id", "src", "path", "family", "n", "graph_seed", "coords", "mel"]
recs = REG.reindex(columns=cols).to_dict("records")
if all(os.path.exists(f"{CACHE}/imgs_{s}.npy") for s in SIZES) and os.path.exists(f"{CACHE}/registry.parquet") and len(pd.read_parquet(f"{CACHE}/registry.parquet")) == len(REG):
    print("cache exists")
else:
    out = Parallel(n_jobs=N_CPU, batch_size=64, verbose=1)(delayed(load_one)(r) for r in recs)
    for k, s in enumerate(SIZES): np.save(f"{CACHE}/imgs_{s}.npy", np.stack([o[0][k] for o in out]))
    REG["mel"] = [o[1] for o in out]
    np.save(f"{CACHE}/img_metrics.npy", REG[M5].to_numpy(np.float32))
    REG.drop(columns=["coords"]).to_parquet(f"{CACHE}/registry.parquet", index=False)
REG = pd.read_parquet(f"{CACHE}/registry.parquet"); MET = np.load(f"{CACHE}/img_metrics.npy")
assert np.isfinite(MET).all(), "non-finite metrics"
print(len(REG), "images cached", {s: os.path.getsize(f"{CACHE}/imgs_{s}.npy") // 2**20 for s in SIZES}, "MB |", f"{(time.time()-T0)/60:.1f} min")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done 2696 tasks      | elapsed:   55.3s
[Parallel(n_jobs=4)]: Done 12296 tasks      | elapsed:  4.2min
[Parallel(n_jobs=4)]: Done 16110 out of 16117 | elapsed:  5.7min remaining:    0.1s
[Parallel(n_jobs=4)]: Done 16117 out of 16117 | elapsed:  5.7min finished


16117 images cached {224: 771, 320: 1573} MB | 6.1 min


## 2. Training pairs and evaluation pairs

In [4]:
IDX = dict(zip(REG.drawing_id, range(len(REG))))
def with_d(df):
    df = df.copy(); df["A_idx"] = df.drawing_A.map(IDX).astype(int); df["B_idx"] = df.drawing_B.map(IDX).astype(int)
    for k, m in enumerate(M5): df["d_" + m] = MET[df.B_idx, k] - MET[df.A_idx, k]      # positive = B higher (A lower stress for ksm)
    return df
L = with_d(LJP)
TRAIN = L[(L.split == "train") & ((L.kind == "natural") | L.in_bal)][["pair_id", "kind", "A_idx", "B_idx", "d_ksm"]]
TRAIN.to_parquet(f"{CACHE}/train_pairs.parquet", index=False)
ev = []
for sp in ["val", "test"]:
    ev.append(L[(L.split == sp) & (L.kind == "natural")].assign(eval_set=f"lj_nat_{sp}"))
    ev.append(L[(L.split == sp) & L.in_bal].assign(eval_set=f"lj_pdsg_{sp}"))
ev.append(L[(L.split == "test") & (L.kind == "pdsg") & ~L.in_bal].assign(eval_set="lj_pdsg_extra_test"))
P3 = with_d(PR); P3["eval_set"] = P3.set.map({"main": "n03_main", "e1b_mooney": "n03_e1b", "counterfactual": "n03_cf"})
P6 = with_d(MCP).assign(eval_set="n06_cf")
KEEP = ["pair_id", "eval_set", "A_idx", "B_idx"] + ["d_" + m for m in M5] + ["cond", "dstress_design", "family", "n", "graph", "base_pair", "cf_proxy", "cf_delta"]
EV = pd.concat([d.reindex(columns=KEEP) for d in ev + [P3, P6]], ignore_index=True)
for c in ["cond", "family", "base_pair", "cf_proxy"]: EV[c] = EV[c].astype(object).where(EV[c].notna(), None)
for c in ["dstress_design", "n", "graph", "cf_delta"]: EV[c] = pd.to_numeric(EV[c], errors="coerce")
assert EV.pair_id.is_unique
EV.to_parquet(f"{CACHE}/eval_pairs.parquet", index=False); EV.to_parquet(f"{OUTPUT_DIR}/eval_pairs_meta.parquet", index=False)
print("train:", TRAIN.kind.value_counts().to_dict()); print("eval:", EV.eval_set.value_counts().to_dict())

train: {'natural': 7573, 'pdsg': 3983}
eval: {'n03_cf': 2839, 'n03_main': 1926, 'n06_cf': 1546, 'lj_nat_val': 1316, 'lj_nat_test': 1282, 'lj_pdsg_test': 721, 'lj_pdsg_val': 700, 'n03_e1b': 289, 'lj_pdsg_extra_test': 183}


## 3. Run configurations (ordered by priority: seed-0 main grid and ablations first, extra seeds last)

In [5]:
pn = lambda p: f"p{int(round(p * 100)):03d}"
def cfg(name, group, backbone="convnext_t", p=0.0, seed=0, **kw):
    c = dict(name=name, group=group, backbone=backbone, p_pdsg=p, seed=seed, iz=IZ, lam_aux=LAM_AUX, img=IMG, epochs=EPOCHS,
             epoch_pairs=EPOCH_PAIRS, batch=BATCH, lr_head=LR_HEAD, lr_bb=None, frozen=False, tta=TTA, pretrained=PRETRAINED, save_ckpt=False)
    c.update(kw); return c
CONFIGS = [cfg(f"cnx_{pn(p)}_s0", "main", p=p, save_ckpt=p in (0.0, 0.3)) for p in MIXES]
CONFIGS += [cfg(f"dino_{pn(p)}_s0", "dinov2", "dinov2_s", p) for p in ABL_MIXES]
CONFIGS += [cfg(f"r18_{pn(p)}_s0", "resnet18", "resnet18", p) for p in ABL_MIXES]
CONFIGS += [cfg(f"cnx_p030_iz{int(round(z*100)):02d}_s0", f"iz{z}", p=0.3, iz=z) for z in IZ_ABL]
CONFIGS += [cfg(f"cnx_{pn(p)}_aux0_s0", "no_aux", p=p, lam_aux=0.0) for p in ABL_MIXES]
CONFIGS += [cfg(f"dinofrz_{pn(p)}_s0", "dinov2_frozen", "dinov2_s", p, frozen=True) for p in ABL_MIXES]
CONFIGS += [cfg(f"cnx_{pn(p)}_img320_s0", "img320", p=p, img=320, batch=32) for p in ABL_MIXES]
for s in SEEDS[1:]:
    CONFIGS += [cfg(f"cnx_{pn(p)}_s{s}", "main", p=p, seed=s) for p in MIXES]
    CONFIGS += [cfg(f"dino_{pn(p)}_s{s}", "dinov2", "dinov2_s", p, s) for p in ABL_MIXES]
if FAST:
    CONFIGS = [cfg("cnx_p000_s0", "main", p=0.0, save_ckpt=True), cfg("cnx_p030_s0", "main", p=0.3),
               cfg("dino_p030_s0", "dinov2", "dinov2_s", 0.3), cfg("dinofrz_p000_s0", "dinov2_frozen", "dinov2_s", 0.0, frozen=True)]
    for c in CONFIGS: c.update(epochs=1, epoch_pairs=64, batch=16, tta=2, pretrained=False)
assert len({c["name"] for c in CONFIGS}) == len(CONFIGS)
json.dump(CONFIGS, open(f"{OUTPUT_DIR}/configs.json", "w"), indent=1)
if RESUME_ROOT:
    for sub in ["runs", "preds", "ckpt"]:
        for f in glob.glob(f"{RESUME_ROOT}/**/{sub}/*", recursive=True):
            if not os.path.exists(f"{OUTPUT_DIR}/{sub}/{os.path.basename(f)}"): shutil.copy(f, f"{OUTPUT_DIR}/{sub}/")
done = lambda c: os.path.exists(f"{OUTPUT_DIR}/runs/{c['name']}.json") and os.path.exists(f"{OUTPUT_DIR}/preds/{c['name']}.parquet")
print(len(CONFIGS), "runs |", sum(map(done, CONFIGS)), "already done")

31 runs | 0 already done


## 4. Training worker

In [6]:
%%writefile /kaggle/working/n09_output/lj_train.py
"""
lj_train.py — LiteJudge training/evaluation worker (PDCP, N09). One configuration per process.

LiteJudge = Siamese image model. A shared backbone maps each drawing to a scalar stress score s(x)
(higher = more stress). For a pair (A, B): u = s(B) - s(A)  (u > 0  ->  A has lower stress).
u is antisymmetric by construction, so the judge is order-free and transitive.
Ordinal head (CORAL, 3 ordered classes B-lower < same < A-lower) with a learned symmetric margin tau:
    P(y >= 1) = sigmoid(u + tau),   P(y >= 2) = sigmoid(u - tau).
Auxiliary heads regress the five layout metrics (ksm, nu, eld, xr, mel; z-scored) of each image (weight lam_aux).

Usage: python lj_train.py --cfg '<json>' --cache /tmp/lj_cache --out /kaggle/working/n09_output
"""
import argparse, json, math, os, time
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

BACKBONES = {   # name -> (timm id, default backbone lr)
    "convnext_t": ("convnext_tiny.fb_in22k_ft_in1k", 1e-4),
    "dinov2_s": ("vit_small_patch14_dinov2.lvd142m", 3e-5),
    "resnet18": ("resnet18.a1_in1k", 3e-4),
}
METRICS = ["ksm", "nu", "eld", "xr", "mel"]


class LiteJudge(nn.Module):
    def __init__(self, backbone, img, pretrained=True, frozen=False):
        super().__init__()
        import timm
        tid = BACKBONES[backbone][0]
        kw = {"img_size": img} if "dinov2" in tid else {}
        self.bb = timm.create_model(tid, pretrained=pretrained, num_classes=0, **kw)
        d = self.bb.num_features
        self.score = nn.Sequential(nn.Linear(d, 256), nn.GELU(), nn.Linear(256, 1))
        self.aux = nn.Linear(d, len(METRICS))
        self.t = nn.Parameter(torch.tensor(-1.0))
        self.frozen = frozen
        if frozen:
            for p in self.bb.parameters():
                p.requires_grad = False
        cfg = getattr(self.bb, "pretrained_cfg", {}) or {}
        self.register_buffer("mean", torch.tensor(cfg.get("mean", (0.485, 0.456, 0.406))).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(cfg.get("std", (0.229, 0.224, 0.225))).view(1, 3, 1, 1))

    def tau(self):
        return F.softplus(self.t) + 1e-3

    def forward(self, x):                      # x: uint8-derived float in [0,1], shape (B,1,H,W)
        x = (x.expand(-1, 3, -1, -1) - self.mean) / self.std
        if self.frozen:
            self.bb.eval()
            with torch.no_grad():
                h = self.bb(x)
        else:
            h = self.bb(x)
        h = h.float()
        return self.score(h).squeeze(-1), self.aux(h)


def dihedral(x, k, f):
    y = torch.rot90(x, int(k), dims=(2, 3))
    return torch.flip(y, dims=(3,)) if f else y


def augment(x, gen):
    """Independent random dihedral transform per image (all layout metrics are invariant to it)."""
    k = torch.randint(0, 4, (x.shape[0],), generator=gen)
    f = torch.randint(0, 2, (x.shape[0],), generator=gen)
    out = torch.empty_like(x)
    for kk in range(4):
        for ff in range(2):
            m = ((k == kk) & (f == ff)).to(x.device)
            if m.any():
                out[m] = dihedral(x[m], kk, ff)
    return out


def coral_loss(u, y, tau):
    return (F.binary_cross_entropy_with_logits(u + tau, (y >= 1).float())
            + F.binary_cross_entropy_with_logits(u - tau, (y >= 2).float()))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cfg", required=True); ap.add_argument("--cache", required=True); ap.add_argument("--out", required=True)
    a = ap.parse_args()
    C = json.loads(a.cfg)
    name = C["name"]
    os.makedirs(f"{a.out}/preds", exist_ok=True); os.makedirs(f"{a.out}/runs", exist_ok=True)
    log = open(f"{a.out}/runs/{name}.log", "a")
    def P(s):
        line = f"[{name} {time.strftime('%H:%M:%S')}] {s}"; print(line, flush=True); log.write(line + "\n"); log.flush()
    t_start = time.time()
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = dev.type == "cuda"
    torch.manual_seed(C["seed"]); np.random.seed(C["seed"])
    gen = torch.Generator().manual_seed(C["seed"])

    imgs = np.load(f"{a.cache}/imgs_{C['img']}.npy", mmap_mode="r")
    met = np.load(f"{a.cache}/img_metrics.npy")
    TR = pd.read_parquet(f"{a.cache}/train_pairs.parquet")
    EV = pd.read_parquet(f"{a.cache}/eval_pairs.parquet")
    mu, sd = met[np.unique(np.r_[TR.A_idx, TR.B_idx])].mean(0), met[np.unique(np.r_[TR.A_idx, TR.B_idx])].std(0) + 1e-6
    metz = torch.tensor((met - mu) / sd, dtype=torch.float32)
    nat = TR[TR.kind == "natural"].reset_index(drop=True)
    pds = TR[TR.kind == "pdsg"].reset_index(drop=True)
    VAL = EV[EV.eval_set.isin(["lj_nat_val", "lj_pdsg_val"])].reset_index(drop=True)

    def batch_x(idx):
        x = torch.from_numpy(np.ascontiguousarray(imgs[np.asarray(idx)])).to(dev, non_blocking=True)
        return x.unsqueeze(1).float().div_(255.0)

    model = LiteJudge(C["backbone"], C["img"], C.get("pretrained", True), C.get("frozen", False)).to(dev)
    lr_bb = C.get("lr_bb") or BACKBONES[C["backbone"]][1]
    head = list(model.score.parameters()) + list(model.aux.parameters()) + [model.t]
    groups = [{"params": head, "lr": C["lr_head"]}]
    if not C.get("frozen", False):
        groups.append({"params": [p for p in model.bb.parameters() if p.requires_grad], "lr": lr_bb})
    opt = torch.optim.AdamW(groups, weight_decay=0.05)
    base_lr = [g["lr"] for g in opt.param_groups]
    steps_per_epoch = math.ceil(C["epoch_pairs"] / C["batch"])
    total = steps_per_epoch * C["epochs"]; warm = steps_per_epoch
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    P(f"{C['backbone']} p_pdsg={C['p_pdsg']} seed={C['seed']} img={C['img']} | natural {len(nat)} pdsg {len(pds)} | {total} steps on {dev}")

    @torch.no_grad()
    def score_images(idx_list, tta=1):
        model.eval()
        idx_list = np.asarray(idx_list); out = np.zeros(len(idx_list), dtype=np.float64)
        tr = [(k, f) for f in (0, 1) for k in range(4)][:tta]
        for s in range(0, len(idx_list), 128):
            x = batch_x(idx_list[s:s + 128]); acc = 0
            for k, f in tr:
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    sc, _ = model(dihedral(x, k, f))
                acc = acc + sc.float()
            out[s:s + 128] = (acc / len(tr)).cpu().numpy()
        model.train()
        return out

    def pair_eval(df, tta):
        ids = np.unique(np.r_[df.A_idx, df.B_idx]); s = dict(zip(ids, score_images(ids, tta)))
        sA = df.A_idx.map(s).values; sB = df.B_idx.map(s).values
        return sA, sB, sB - sA

    curve = []; step = 0; rng = np.random.default_rng(C["seed"])
    for ep in range(C["epochs"]):
        t0 = time.time()
        n_p = int(round(C["p_pdsg"] * C["epoch_pairs"])); n_n = C["epoch_pairs"] - n_p
        parts = []
        if n_n: parts.append(nat.iloc[rng.choice(len(nat), n_n, replace=n_n > len(nat))])
        if n_p: parts.append(pds.iloc[rng.choice(len(pds), n_p, replace=n_p > len(pds))])
        E = pd.concat(parts).sample(frac=1, random_state=int(rng.integers(1e9))).reset_index(drop=True)
        d = E.d_ksm.values
        y = np.where(d > C["iz"], 2, np.where(d < -C["iz"], 0, 1))
        swap = rng.random(len(E)) < 0.5                     # random A/B order
        A = np.where(swap, E.B_idx, E.A_idx); B = np.where(swap, E.A_idx, E.B_idx); y = np.where(swap, 2 - y, y)
        tl, tn = 0.0, 0
        for s in range(0, len(E), C["batch"]):
            lr_f = (step + 1) / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            for g, b in zip(opt.param_groups, base_lr): g["lr"] = b * lr_f
            ia, ib = A[s:s + C["batch"]], B[s:s + C["batch"]]
            x = augment(batch_x(np.r_[ia, ib]), gen)
            yt = torch.tensor(y[s:s + C["batch"]], device=dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                sc, ax = model(x)
            sc, ax = sc.float(), ax.float()
            nb = len(ia)
            u = sc[nb:] - sc[:nb]
            loss = coral_loss(u, yt, model.tau())
            if C["lam_aux"] > 0:
                loss = loss + C["lam_aux"] * F.mse_loss(ax, metz[np.r_[ia, ib]].to(dev))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); step += 1
            tl += loss.item() * nb; tn += nb
        _, _, uv = pair_eval(VAL, 1)
        m = VAL.d_ksm.abs().values >= 0.02
        accs = {k: float(((uv > 0) == (VAL.d_ksm.values > 0))[m & (VAL.eval_set == k).values].mean()) for k in ["lj_nat_val", "lj_pdsg_val"]}
        curve.append({"epoch": ep + 1, "loss": tl / tn, "val_nat_acc": accs["lj_nat_val"], "val_pdsg_acc": accs["lj_pdsg_val"],
                      "tau": float(model.tau()), "sec": round(time.time() - t0, 1)})
        P(json.dumps(curve[-1]))

    # final evaluation (last epoch, no model selection) with dihedral test-time averaging
    t0 = time.time()
    sA, sB, u = pair_eval(EV, C.get("tta", 8))
    tau = float(model.tau())
    out = EV[["pair_id", "eval_set"]].copy()
    out["sA"], out["sB"], out["u"] = sA, sB, u
    out["p_A_lower"] = 1 / (1 + np.exp(-(u - tau))); out["p_same"] = 1 / (1 + np.exp(-(u + tau))) - out.p_A_lower
    out["p_A_2afc"] = 1 / (1 + np.exp(-u))
    out.to_parquet(f"{a.out}/preds/{name}.parquet", index=False)
    if C.get("save_ckpt"):
        os.makedirs(f"{a.out}/ckpt", exist_ok=True)
        torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}, f"{a.out}/ckpt/{name}.pt")
    res = {"cfg": C, "curve": curve, "tau": tau, "eval_sec": round(time.time() - t0, 1), "train_min": round((time.time() - t_start) / 60, 2),
           "vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if amp else None}
    json.dump(res, open(f"{a.out}/runs/{name}.json", "w"), indent=1)
    P(f"done in {res['train_min']} min")


if __name__ == "__main__":
    main()


Writing /kaggle/working/n09_output/lj_train.py


## 5. Run (one run per GPU at a time; runs are independent and resumable)

In [7]:
DEADLINE = T0 + SESSION_HOURS * 3600
queue = [c for c in CONFIGS if not done(c)]; running = {}; slots = max(1, N_GPU)
def launch(c, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu) if N_GPU else "", PYTHONUNBUFFERED="1", OMP_NUM_THREADS=str(max(1, N_CPU // slots)))
    logf = open(f"{OUTPUT_DIR}/runs/{c['name']}.stdout", "a")
    p = subprocess.Popen([sys.executable, f"{OUTPUT_DIR}/lj_train.py", "--cfg", json.dumps(c), "--cache", CACHE, "--out", OUTPUT_DIR],
                         stdout=logf, stderr=subprocess.STDOUT, env=env)
    running[gpu] = (c, p, logf); print(f"[{(time.time()-T0)/60:6.1f} min] start {c['name']} on GPU {gpu}", flush=True)
while queue or running:
    for gpu in range(slots):
        if gpu not in running and queue and time.time() < DEADLINE - MIN_LEFT_MIN * 60: launch(queue.pop(0), gpu)
    if not running: break
    time.sleep(5 if FAST else 20)
    for gpu, (c, p, logf) in list(running.items()):
        if p.poll() is not None:
            logf.close(); del running[gpu]
            tail = open(f"{OUTPUT_DIR}/runs/{c['name']}.stdout").read().strip().splitlines()
            print(f"[{(time.time()-T0)/60:6.1f} min] end {c['name']} (exit {p.returncode}): {tail[-1][-200:] if tail else ''}", flush=True)
            if p.returncode != 0: print("\n".join(tail[-15:]))
missing = [c["name"] for c in CONFIGS if not done(c)]
print(f"run phase finished after {(time.time()-T0)/3600:.2f} h | missing runs: {missing}")

[   6.1 min] start cnx_p000_s0 on GPU 0
[   6.1 min] start cnx_p010_s0 on GPU 1
[  20.4 min] end cnx_p000_s0 (exit 0): [cnx_p000_s0 00:21:57] done in 13.97 min
[  20.4 min] start cnx_p030_s0 on GPU 0
[  20.7 min] end cnx_p010_s0 (exit 0): [cnx_p010_s0 00:22:18] done in 14.33 min
[  20.7 min] start cnx_p050_s0 on GPU 1
[  34.4 min] end cnx_p030_s0 (exit 0): [cnx_p030_s0 00:36:09] done in 13.83 min
[  34.4 min] start cnx_p100_s0 on GPU 0
[  35.4 min] end cnx_p050_s0 (exit 0): [cnx_p050_s0 00:36:56] done in 14.3 min
[  35.4 min] start dino_p000_s0 on GPU 1
[  48.4 min] end cnx_p100_s0 (exit 0): [cnx_p100_s0 00:50:12] done in 13.89 min
[  48.4 min] start dino_p030_s0 on GPU 0
[  49.1 min] end dino_p000_s0 (exit 0): [dino_p000_s0 00:50:47] done in 13.47 min
[  49.1 min] start r18_p000_s0 on GPU 1
[  53.7 min] end r18_p000_s0 (exit 0): [r18_p000_s0 00:55:32] done in 4.56 min
[  53.7 min] start r18_p030_s0 on GPU 1
[  58.4 min] end r18_p030_s0 (exit 0): [r18_p030_s0 01:00:11] done in 4.54 min

## 6. Analysis

In [8]:
EVM = pd.read_parquet(f"{OUTPUT_DIR}/eval_pairs_meta.parquet")
CF = {c["name"]: c for c in CONFIGS}
RUNS = {c["name"]: json.load(open(f"{OUTPUT_DIR}/runs/{c['name']}.json")) for c in CONFIGS if done(c)}
PRED = {n: pd.read_parquet(f"{OUTPUT_DIR}/preds/{n}.parquet").merge(EVM.drop(columns="eval_set"), on="pair_id") for n in RUNS}
CUES = ["nu", "eld", "xr", "mel"]
def acc(d):
    m = d.d_ksm.abs() >= 0.02; return float(((d.u[m] > 0) == (d.d_ksm[m] > 0)).mean()) if m.any() else np.nan
def acc3(d, tau):
    y = np.where(d.d_ksm > 0.02, 2, np.where(d.d_ksm < -0.02, 0, 1))
    pr = np.stack([1 - d.p_A_lower - d.p_same, d.p_same, d.p_A_lower], 1).argmax(1); return float((pr == y).mean())
def betas(d):
    X = d[["d_" + m for m in ["ksm"] + CUES]].to_numpy(float); X = (X - X.mean(0)) / (X.std(0) + 1e-12)
    y = (d.u - d.u.mean()) / (d.u.std() + 1e-12)
    b = np.linalg.lstsq(np.c_[np.ones(len(d)), X], y, rcond=None)[0][1:]; return dict(zip(["ksm"] + CUES, b))
def share(b): c = sum(abs(b[k]) for k in CUES); return c / (c + abs(b["ksm"]))
def lure(d):
    z = d[d.dstress_design == 0]; a = z[z.cond == "congruent"]; b = z[z.cond == "conflict_all"]
    return float((a.u > 0).mean() - (b.u > 0).mean()) if len(a) and len(b) else np.nan
def ccgap(d):
    nz = d[d.dstress_design > 0]; return acc(nz[nz.cond == "congruent"]) - acc(nz[nz.cond.astype(str).str.startswith("conflict")])
def cf_table(Pd):
    base = Pd[Pd.eval_set == "n03_main"].set_index("pair_id")
    cf = Pd[Pd.eval_set.isin(["n03_cf", "n06_cf"]) & Pd.base_pair.isin(base.index)].copy()
    cf["shift"] = cf.p_A_2afc.values - cf.base_pair.map(base.p_A_2afc).values
    cf["flip"] = (cf.u.values > 0) != (cf.base_pair.map(base.u).values > 0)
    return cf.groupby(["cf_proxy", "cf_delta"]).agg(n=("shift", "size"), shift=("shift", "mean"), shift_se=("shift", lambda s: s.std() / np.sqrt(len(s))),
                                                   flip=("flip", "mean")).reset_index()
def img_rho(Pd, sets=("lj_nat_test", "lj_pdsg_test")):
    d = Pd[Pd.eval_set.isin(sets)]
    S = pd.concat([d[["A_idx", "sA", "family", "n", "graph"]].set_axis(["idx", "s", "family", "n", "graph"], axis=1),
                   d[["B_idx", "sB", "family", "n", "graph"]].set_axis(["idx", "s", "family", "n", "graph"], axis=1)]).drop_duplicates("idx")
    S["ksm"], S["mel"] = MET[S.idx, 0], MET[S.idx, 4]
    r = S.groupby(["family", "n", "graph"]).apply(lambda g: pd.Series({"ksm": g.s.corr(g.ksm, method="spearman"), "mel": g.s.corr(g.mel, method="spearman")}) if len(g) > 4 else pd.Series({"ksm": np.nan, "mel": np.nan}))
    return r.mean().to_dict()
rows, brows, cfrows = [], [], []
for n, Pd in PRED.items():
    c = CF[n]; G = {k: g for k, g in Pd.groupby("eval_set")}; r = {k: c[k] for k in ["name", "group", "backbone", "p_pdsg", "seed", "iz", "lam_aux", "img", "frozen"]}
    for k in ["lj_nat_test", "lj_pdsg_test", "lj_pdsg_extra_test", "n03_main", "n03_e1b"]:
        if k in G: r["acc_" + k] = acc(G[k])
    if "lj_pdsg_test" in G: r["acc3_lj_pdsg_test"] = acc3(G["lj_pdsg_test"], RUNS[n]["tau"])
    for k in ["lj_pdsg_test", "n03_main"]:
        if k in G and len(G[k]) > 10:
            b = betas(G[k]); r["cue_share_" + k] = share(b); r["lure_" + k] = lure(G[k]); r["ccgap_" + k] = ccgap(G[k])
            brows.append({"name": n, "eval_set": k, **{"beta_" + q: v for q, v in b.items()}})
    t = cf_table(Pd)
    if len(t):
        t.insert(0, "name", n); cfrows.append(t)
        null = t[t.cf_proxy == "null"]; r["flip_null"] = float(null.flip.mean()) if len(null) else np.nan
        for px in ["nu", "eld", "xr", "mel"]:
            tt = t[t.cf_proxy == px]; r[f"flip_{px}"] = float(np.average(tt.flip, weights=tt.n)) if len(tt) else np.nan
    rr = img_rho(Pd); r["rho_score_ksm"], r["rho_score_mel"] = rr["ksm"], rr["mel"]
    r["tau"] = RUNS[n]["tau"]; r["train_min"] = RUNS[n]["train_min"]; rows.append(r)
RUNT = pd.DataFrame(rows); BET = pd.DataFrame(brows); CFT = pd.concat(cfrows, ignore_index=True) if cfrows else pd.DataFrame()
RUNT.round(4).to_csv(f"{OUTPUT_DIR}/lj_runs.csv", index=False); BET.round(4).to_csv(f"{OUTPUT_DIR}/lj_betas.csv", index=False)
if len(CFT): CFT.round(4).to_csv(f"{OUTPUT_DIR}/lj_cf_effects.csv", index=False)
num = [c for c in RUNT.columns if c not in ["name", "group", "backbone", "p_pdsg", "seed", "iz", "lam_aux", "img", "frozen"]]
MIX = RUNT.groupby(["group", "backbone", "p_pdsg"])[num].agg(["mean", "std"]); MIX.columns = [f"{a}_{b}" for a, b in MIX.columns]
MIX.insert(0, "n_seeds", RUNT.groupby(["group", "backbone", "p_pdsg"]).size()); MIX = MIX.reset_index()
MIX.round(4).to_csv(f"{OUTPUT_DIR}/lj_mix_summary.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
show = ["group", "backbone", "p_pdsg", "n_seeds"] + [f"{k}_mean" for k in ["acc_lj_nat_test", "acc_lj_pdsg_test", "acc_n03_main", "cue_share_lj_pdsg_test",
        "cue_share_n03_main", "lure_n03_main", "ccgap_n03_main", "flip_null", "flip_mel", "rho_score_ksm"] if f"{k}_mean" in MIX]
print(MIX[show].round(3).to_string(index=False))

/tmp/ipykernel_23/1299783404.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  r = S.groupby(["family", "n", "graph"]).apply(lambda g: pd.Series({"ksm": g.s.corr(g.ksm, method="spearman"), "mel": g.s.corr(g.mel, method="spearman")}) if len(g) > 4 else pd.Series({"ksm": np.nan, "mel": np.nan}))
/tmp/ipykernel_23/1299783404.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  r = S.groupby(["family", "n", "graph"]).

        group   backbone  p_pdsg  n_seeds  acc_lj_nat_test_mean  acc_lj_pdsg_test_mean  acc_n03_main_mean  cue_share_lj_pdsg_test_mean  cue_share_n03_main_mean  lure_n03_main_mean  ccgap_n03_main_mean  flip_null_mean  flip_mel_mean  rho_score_ksm_mean
       dinov2   dinov2_s     0.0        3                 0.998                  0.676              0.921                        0.616                    0.665               0.522                0.110           0.088          0.175               0.807
       dinov2   dinov2_s     0.3        3                 0.996                  0.966              0.984                        0.216                    0.200              -0.106                0.017           0.065          0.090               0.922
dinov2_frozen   dinov2_s     0.0        1                 0.996                  0.771              0.951                        0.514                    0.559               0.376                0.065           0.086          0.150             

/tmp/ipykernel_23/1299783404.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  r = S.groupby(["family", "n", "graph"]).apply(lambda g: pd.Series({"ksm": g.s.corr(g.ksm, method="spearman"), "mel": g.s.corr(g.mel, method="spearman")}) if len(g) > 4 else pd.Series({"ksm": np.nan, "mel": np.nan}))


## 7. Figures

In [9]:
bm, pv = None, pd.DataFrame()
main = RUNT[RUNT.group == "main"]; agg = main.groupby("p_pdsg"); P_ = sorted(main.p_pdsg.unique()); x = np.array(P_) * 100
def band(ax, col, label, **kw):
    if col not in main: return
    m, s = agg[col].mean().reindex(P_), agg[col].std().reindex(P_).fillna(0)
    kw.setdefault("marker", "o"); ax.errorbar(x, m, yerr=s, capsize=3, label=label, **kw)
fig, ax = plt.subplots(1, 4, figsize=(20, 4.2))
band(ax[0], "acc_lj_nat_test", "natural pairs (LJ test)"); band(ax[0], "acc_lj_pdsg_test", "decoupled pairs (LJ test)"); band(ax[0], "acc_n03_main", "VLM eval pairs (N03 main)")
ax[0].axhline(0.5, ls=":", c="grey"); ax[0].set_title("Accuracy (|Δstress| ≥ 0.02)"); ax[0].set_ylim(0.4, 1.0)
band(ax[1], "cue_share_lj_pdsg_test", "decoupled pairs (LJ test)"); band(ax[1], "cue_share_n03_main", "VLM eval pairs (N03 main)")
ax[1].set_title("Cue share = Σ|β cue| / (Σ|β cue| + |β stress|)"); ax[1].set_ylim(0, 1)
band(ax[2], "lure_lj_pdsg_test", "decoupled pairs (LJ test)"); band(ax[2], "lure_n03_main", "VLM eval pairs (N03 main)")
ax[2].axhline(0, c="k", lw=.8); ax[2].set_title("Lure gap at Δstress = 0 (congruent − conflict_all)")
for px, mk in [("nu", "s"), ("eld", "^"), ("xr", "v"), ("mel", "D"), ("null", "x")]:
    band(ax[3], f"flip_{px}", f"{px} edit", marker=mk) if px != "null" else band(ax[3], "flip_null", "null edit (noise floor)", ls="--", c="k")
ax[3].set_title("Counterfactual flip rate (stress unchanged)")
for a in ax: a.set_xlabel("PDSG share of training pairs (%)"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_lj_mix.png", dpi=130); plt.show()

if len(BET):
    b = BET.merge(RUNT[["name", "group", "p_pdsg"]], on="name"); b = b[(b.group == "main") & (b.eval_set == "n03_main")]
    bm = b.groupby("p_pdsg")[[f"beta_{k}" for k in ["ksm"] + CUES]].mean()
    if len(bm) == 0: bm = None
if len(BET) and bm is not None:
    bm.columns = ["stress"] + [c.upper() for c in CUES]; bm.index = [f"{int(p*100)}% PDSG" for p in bm.index]
    ax = bm.T.plot.bar(figsize=(10, 4)); ax.axhline(0, c="k", lw=.8); ax.set_ylabel("standardized weight on u"); ax.set_title("What drives LiteJudge's decision on the VLM evaluation pairs (N03 main)")
    plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_lj_betas.png", dpi=130); plt.show()

pick = [n for n in ["cnx_p000_s0", "cnx_p030_s0", "cnx_p100_s0"] if n in PRED]
if pick:
    fig, ax = plt.subplots(1, len(pick), figsize=(5.5 * len(pick), 4), sharey=True); ax = np.atleast_1d(ax)
    for a, n in zip(ax, pick):
        d = PRED[n][PRED[n].eval_set == "n03_main"]
        for cnd, g in d.groupby("cond"):
            pc = g.groupby("dstress_design").apply(lambda h: (h.u > 0).mean())
            a.plot(pc.index, pc.values, marker="o", label=cnd, lw=2.5 if cnd in ("congruent", "conflict_all") else 1)
        a.axhline(0.5, ls=":", c="grey"); a.set_title(f"{n}: P(choose lower-stress A)"); a.set_xlabel("Δstress (design)")
    ax[0].set_ylabel("P(choose A)"); ax[-1].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_lj_psychometric.png", dpi=130); plt.show()

ab = RUNT[RUNT.seed == 0].copy(); ab["label"] = ab.group + " · " + (ab.p_pdsg * 100).astype(int).astype(str) + "%"
ab = ab.sort_values(["p_pdsg", "group"])
fig, ax = plt.subplots(1, 3, figsize=(19, max(4, 0.32 * len(ab))))
for a, col, t in zip(ax, ["acc_lj_pdsg_test", "acc_n03_main", "cue_share_n03_main"], ["Accuracy, decoupled LJ test", "Accuracy, N03 main", "Cue share, N03 main"]):
    if col in ab: a.barh(ab.label, ab[col], color=["C0" if p == 0 else "C1" for p in ab.p_pdsg]); a.set_title(t)
ax[0].invert_yaxis(); ax[1].invert_yaxis(); ax[2].invert_yaxis()
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_lj_ablation.png", dpi=130); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for n in [f"cnx_{pn(p)}_s0" for p in MIXES if f"cnx_{pn(p)}_s0" in RUNS]:
    cv = pd.DataFrame(RUNS[n]["curve"]); ax[0].plot(cv.epoch, cv.val_nat_acc, marker=".", label=n); ax[1].plot(cv.epoch, cv.val_pdsg_acc, marker=".", label=n)
ax[0].set_title("Validation accuracy, natural pairs"); ax[1].set_title("Validation accuracy, decoupled pairs")
for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_lj_curves.png", dpi=130); plt.show()

if len(CFT):
    t = CFT.merge(RUNT[["name", "group", "p_pdsg"]], on="name"); t = t[t.group == "main"]
    t["edit"] = t.cf_proxy + " " + t.cf_delta.map(lambda v: f"{v:+.2f}")
    pv = t.groupby(["edit", "p_pdsg"])["shift"].mean().unstack()
if len(CFT) and len(pv):
    ax = pv.plot.barh(figsize=(10, max(4, 0.35 * len(pv)))); ax.axvline(0, c="k", lw=.8)
    ax.set_xlabel("mean shift in P(choose edited drawing A) — stress unchanged"); ax.set_title("Counterfactual sensitivity of LiteJudge by PDSG share")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_lj_cf.png", dpi=130); plt.show()

/tmp/ipykernel_23/2802979797.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pc = g.groupby("dstress_design").apply(lambda h: (h.u > 0).mean())
/tmp/ipykernel_23/2802979797.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pc = g.groupby("dstress_design").apply(lambda h: (h.u > 0).mean())
/tmp/ipykernel_23/2802979797.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior 

## 8. Report and manifest

In [10]:
def mm(col, p):
    v = main[main.p_pdsg == p][col] if col in main else pd.Series(dtype=float); return [round(float(v.mean()), 4), round(float(v.std()), 4) if len(v) > 1 else None, int(len(v))]
REP = {"runtime_h": round((time.time() - T0) / 3600, 2), "gpus": N_GPU, "runs_done": len(RUNS), "runs_total": len(CONFIGS),
       "missing": [c["name"] for c in CONFIGS if not done(c)], "train_pairs": TRAIN.kind.value_counts().to_dict(),
       "eval_pairs": EV.eval_set.value_counts().to_dict(),
       "main_by_pdsg_share": {f"{int(p*100)}%": {k: mm(k, p) for k in ["acc_lj_nat_test", "acc_lj_pdsg_test", "acc_n03_main", "acc_n03_e1b",
                                "cue_share_lj_pdsg_test", "cue_share_n03_main", "lure_n03_main", "ccgap_n03_main", "flip_null", "flip_mel", "rho_score_ksm", "rho_score_mel"]}
                              for p in sorted(main.p_pdsg.unique())},
       "note": "values are [mean, sd over seeds, n_seeds]; accuracy on pairs with |Δstress| >= 0.02; last epoch, no model selection"}
json.dump(REP, open(f"{OUTPUT_DIR}/n09_report.json", "w"), indent=1, default=str)
print(json.dumps(REP["main_by_pdsg_share"], indent=1)[:3000])
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N09 LiteJudge · {REP['runtime_h']} h · runs {len(RUNS)}/{len(CONFIGS)} · missing: {REP['missing']}", ""]
for root, _, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        p = os.path.join(root, f); lines.append(f"{os.path.relpath(p, OUTPUT_DIR):45s} {os.path.getsize(p)/1024:10.1f} KB")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines[:60]))
print("ALL COMPLETE" if not REP["missing"] else "NOT COMPLETE -> save output as dataset, set RESUME_ROOT, run again")

{
 "0%": {
  "acc_lj_nat_test": [
   0.9997,
   0.0005,
   3
  ],
  "acc_lj_pdsg_test": [
   0.8492,
   0.0076,
   3
  ],
  "acc_n03_main": [
   0.9292,
   0.0023,
   3
  ],
  "acc_n03_e1b": [
   0.5882,
   0.0104,
   3
  ],
  "cue_share_lj_pdsg_test": [
   0.5179,
   0.022,
   3
  ],
  "cue_share_n03_main": [
   0.5038,
   0.0105,
   3
  ],
  "lure_n03_main": [
   0.5791,
   0.0339,
   3
  ],
  "ccgap_n03_main": [
   0.1017,
   0.0035,
   3
  ],
  "flip_null": [
   0.0776,
   0.0153,
   3
  ],
  "flip_mel": [
   0.1208,
   0.0046,
   3
  ],
  "rho_score_ksm": [
   0.8663,
   0.0055,
   3
  ],
  "rho_score_mel": [
   0.8078,
   0.0048,
   3
  ]
 },
 "10%": {
  "acc_lj_nat_test": [
   1.0,
   0.0,
   3
  ],
  "acc_lj_pdsg_test": [
   0.989,
   0.0,
   3
  ],
  "acc_n03_main": [
   0.9847,
   0.0042,
   3
  ],
  "acc_n03_e1b": [
   0.654,
   0.0261,
   3
  ],
  "cue_share_lj_pdsg_test": [
   0.1388,
   0.0016,
   3
  ],
  "cue_share_n03_main": [
   0.1108,
   0.0246,
   3
  ],
  "lure_n0